## Auditing a notebook file's source order

Markdown gaps and IPython magic calls must not be mistaken for executable Python context.

In [1]:
from astscribe import NotebookAnalyzer

document = {
    "cells": [
        {"cell_type": "markdown", "source": "# Experiment"},
        {"cell_type": "code", "source": "prediction = model(1)"},
        {"cell_type": "code", "source": "%timeit model(1)"},
        {"cell_type": "code", "source": "model = object()"},
    ]
}
audit = NotebookAnalyzer.from_ipynb_data(document)
print("Analyzed indices:", audit.cell_indices)
print("Skipped indices:", [cell.index for cell in audit.skipped_cells])

Analyzed indices: (1, 3)
Skipped indices: [2]


The use of `model` in cell 1 precedes its definition in cell 3, so ASTscribe flags a forward reference.

In [2]:
print([(item.code, item.cell, item.related_cell) for item in audit.diagnostics().items])

[('dependency.forward_reference', 1, 3)]


When adding a cell, ASTscribe now continues from the *original notebook index* rather than the number of analyzed cells.

In [3]:
audit.add_cell("result = model(2)")
print("Indices after append:", audit.cell_indices)
print(audit.render_dependency_graph())

Indices after append: (1, 3, 4)
# Cell dependency graph

Cell 3 -> Cell 4 [model]
